## NB01-Data Collection

### Do Popularity and Audience Ratings Move Together Across Film Genres?

Movie audiences don't always reward the same films they claim to enjoy most:
some genres are built for mass appeal, others tend to score higher with the
TMDB user base that rates them. TMDB (The Movie Database) exposes both a
`popularity` score and a `vote_average` per film — the latter is the average
rating left by TMDB's own registered users, not a professional critics' score —
tagged by genre, which makes it possible to check whether the two move together,
and whether that relationship has shifted across decades.

In [1]:
import os
import json
import requests
from dotenv import load_dotenv


load_dotenv()
api_key = os.getenv("API_KEY")

#### Main data source

TMDB's `/movie/popular` endpoint returns films ranked by TMDB's own internal
popularity score. I chose it because it is the only endpoint that exposes
`popularity` directly without needing a separate call per film.

**This is a real limitation, not just a technical choice.** A dataset built from
"already popular" films is not a random sample of movies — it is pre-filtered by
the very variable I'm investigating. This matters directly for my research
question: any comparison I make between popularity and rating is a comparison
*within* the already-popular slice of TMDB, not across all films ever catalogued.
I address this in NB03 by not overinterpreting popularity differences as if they
generalised to obscure or unpopular films.

The alternative, `/discover/movie`, allows filtering by genre or year directly,
but does not return a pre-ranked `popularity` field in the same way and would
require extra requests to reconstruct it. I judged the tradeoff worth it given
the time available, but a future version of this project could combine both.

#### Decision: why 500 pages (10,000 films)

TMDB reports 1,163,851 total movies available (see the printed output below).
10,000 films is roughly 0.86% of the full catalogue — a small slice, but the
right one for this question: it captures the films with enough visibility and
enough votes to make a genre-level popularity/rating comparison meaningful.


In [2]:
url = "https://api.themoviedb.org/3/movie/popular"

n_pages = 500

all_pages = [] 

for page in range(1, n_pages + 1):
    params = {
    "api_key": api_key,
    "language": "en-US",
    "page": page
}
    params["page"] = page
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        page_data = response.json()
        all_pages.append(page_data) 
        
    
print("Data status:", response.status_code)
data = response.json()
with open("../data/raw/movies.json", "w") as f:
    json.dump(all_pages, f)


Data status: 200


How the pagination loop works:

TMDB returns 20 movies per page, so collecting more films means requesting more
pages. This loop calls `/movie/popular` once per page, from 1 to `n_pages`
(500), and keeps only the pages that return a successful response
(`status_code == 200`).

Each request needs its own `params` dictionary because `page` changes on every
iteration — reusing one dictionary across requests would send the same page
number every time.

#### Data structure

In [3]:

print("Overall data type:", type(data))
print("Top-level keys:", data.keys())
print("Keys available per movie:", data["results"][0].keys())
print("Type of 'results':", type(data["results"]))
print("Number of movies on  page 1:", len(data["results"]))
print("Total movies available in TMDB:", data["total_results"])
print("First movie (full record):", data["results"][0])



Overall data type: <class 'dict'>
Top-level keys: dict_keys(['page', 'results', 'total_pages', 'total_results'])
Keys available per movie: dict_keys(['adult', 'backdrop_path', 'genre_ids', 'id', 'title', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'release_date', 'softcore', 'video', 'vote_average', 'vote_count'])
Type of 'results': <class 'list'>
Number of movies on  page 1: 20
Total movies available in TMDB: 1164516
First movie (full record): {'adult': False, 'backdrop_path': '/jPKsE8cgehlau8V6YgNNVCMsoRK.jpg', 'genre_ids': [35, 10749, 10770], 'id': 932430, 'title': 'Prom Pact', 'original_language': 'en', 'original_title': 'Prom Pact', 'overview': 'It\'s prom season, and high school senior Mandy and her best friend and fellow outsider Ben are surrounded by over-the-top "promposals." Mandy is only focused on getting into her dream school Harvard, but as she starts tutoring basketball all-star Graham, she must re-evaluate whether her dream school is 

In [4]:
print("Expected records 200000")
all_movies_check = []
for page_data in all_pages:
    for movie in page_data["results"]:
        all_movies_check.append(movie)

print("Actual records collected:", len(all_movies_check))

Expected records 200000
Actual records collected: 10000


#### Additional data source

`genre_ids` are just numbers. We need a second endpoint to translate those numbers into real genre names (e.g. `28` → `"Action"`) That is why we need to download another json.

In [5]:

url_genres = "https://api.themoviedb.org/3/genre/movie/list"
params_genres = {"api_key": api_key, "language": "en-US"}

response_genres = requests.get(url_genres, params=params_genres)
genres_data = response_genres.json()
with open("../data/raw/genres.json", "w") as f:
    json.dump(genres_data, f)
genres_data

{'genres': [{'id': 28, 'name': 'Action'},
  {'id': 12, 'name': 'Adventure'},
  {'id': 16, 'name': 'Animation'},
  {'id': 35, 'name': 'Comedy'},
  {'id': 80, 'name': 'Crime'},
  {'id': 99, 'name': 'Documentary'},
  {'id': 18, 'name': 'Drama'},
  {'id': 10751, 'name': 'Family'},
  {'id': 14, 'name': 'Fantasy'},
  {'id': 36, 'name': 'History'},
  {'id': 27, 'name': 'Horror'},
  {'id': 10402, 'name': 'Music'},
  {'id': 9648, 'name': 'Mystery'},
  {'id': 10749, 'name': 'Romance'},
  {'id': 878, 'name': 'Science Fiction'},
  {'id': 10770, 'name': 'TV Movie'},
  {'id': 53, 'name': 'Thriller'},
  {'id': 10752, 'name': 'War'},
  {'id': 37, 'name': 'Western'}]}